In [0]:
STORAGE_ACCOUNT = 'stccasemauricioes'
CONTAINER = 'data'
STORAGE_KEY = 'COLOCAR_CHAVE_AQUI'

spark.conf.set("fs.azure.account.key." + STORAGE_ACCOUNT + ".blob.core.windows.net", STORAGE_KEY)

BASE_SILVER = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net/silver"
BASE_GOLD   = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net/gold"

In [0]:
"""
Pipeline Medallion - Camada GOLD
Modelagem dimensional Kimball.
"""
from pyspark.sql import functions as F

print('Carregando Silver...')
pedidos    = spark.read.format('delta').load(f'{BASE_SILVER}/pedidos')
itens      = spark.read.format('delta').load(f'{BASE_SILVER}/pedido_itens')
pagamentos = spark.read.format('delta').load(f'{BASE_SILVER}/pagamentos')
produtos   = spark.read.format('delta').load(f'{BASE_SILVER}/produtos')
lojas      = spark.read.format('delta').load(f'{BASE_SILVER}/lojas')
clientes   = spark.read.format('delta').load(f'{BASE_SILVER}/clientes')

# === dim_data ===
print('Criando dim_data...')
min_date = pedidos.agg(F.min(F.to_date('criado_em'))).collect()[0][0]
max_date = pedidos.agg(F.max(F.to_date('criado_em'))).collect()[0][0]

dim_data = (spark.sql(f"""
    SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS data
""")
.withColumn('data_id',    F.date_format('data', 'yyyyMMdd').cast('int'))
.withColumn('ano',        F.year('data'))
.withColumn('mes',        F.month('data'))
.withColumn('dia',        F.dayofmonth('data'))
.withColumn('dia_semana', F.dayofweek('data'))
.withColumn('nome_dia_semana',
    F.when(F.col('dia_semana') == 1, 'Domingo')
     .when(F.col('dia_semana') == 2, 'Segunda')
     .when(F.col('dia_semana') == 3, 'Terca')
     .when(F.col('dia_semana') == 4, 'Quarta')
     .when(F.col('dia_semana') == 5, 'Quinta')
     .when(F.col('dia_semana') == 6, 'Sexta')
     .otherwise('Sabado'))
.withColumn('eh_fim_de_semana', F.col('dia_semana').isin([1, 7]))
.withColumn('trimestre', F.quarter('data')))

dim_data.write.format('delta').mode('overwrite').save(f'{BASE_GOLD}/dim_data')
print(f'  dim_data: {dim_data.count()}')

# === dim_produto ===
print('Criando dim_produto...')
dim_produto = (produtos.select(
        F.col('sku').alias('produto_id'),
        F.col('nome').alias('produto_nome'),
        F.col('categoria').alias('produto_categoria'),
        F.col('marca').alias('produto_marca'),
        F.col('preco_base').alias('produto_preco_base'),
        F.col('abv').alias('produto_abv'))
    .withColumn('faixa_preco',
        F.when(F.col('produto_preco_base') < 5, 'Economico')
         .when(F.col('produto_preco_base') < 10, 'Padrao')
         .when(F.col('produto_preco_base') < 15, 'Premium')
         .otherwise('Super Premium')))
dim_produto.write.format('delta').mode('overwrite').save(f'{BASE_GOLD}/dim_produto')
print(f'  dim_produto: {dim_produto.count()}')

# === dim_loja ===
dim_loja = lojas.select(
    'loja_id',
    F.col('nome').alias('loja_nome'),
    F.col('cidade').alias('loja_cidade'),
    F.col('estado').alias('loja_estado'),
    F.col('regiao').alias('loja_regiao'))
dim_loja.write.format('delta').mode('overwrite').save(f'{BASE_GOLD}/dim_loja')
print(f'  dim_loja: {dim_loja.count()}')

# === dim_cliente ===
dim_cliente = clientes.select(
    'cliente_id',
    F.col('nome').alias('cliente_nome'),
    F.col('cidade').alias('cliente_cidade'),
    F.to_date('data_cadastro').alias('cliente_data_cadastro'))
dim_cliente.write.format('delta').mode('overwrite').save(f'{BASE_GOLD}/dim_cliente')
print(f'  dim_cliente: {dim_cliente.count()}')

Carregando Silver...
Criando dim_data...
  dim_data: 61
Criando dim_produto...
  dim_produto: 15
  dim_loja: 12
  dim_cliente: 2000


In [0]:
# === fact_vendas ===
# Grao: 1 linha por item de pedido PAGO
print('\nConstruindo fact_vendas...')

pedidos_pagos = pedidos.filter(F.col('status') == 'PAGO')

fact = (itens.alias('i')
    .join(pedidos_pagos.alias('p'),
          F.col('i.pedido_id') == F.col('p.pedido_id'), 'inner')
    .join(pagamentos.alias('pg'),
          F.col('i.pedido_id') == F.col('pg.pedido_id'), 'left')
    .select(
        F.col('i.item_id').alias('item_id'),
        F.col('p.pedido_id').alias('pedido_id'),
        F.col('i.sku').alias('produto_id'),
        F.col('p.loja_id').alias('loja_id'),
        F.col('p.cliente_id').alias('cliente_id'),
        F.date_format(F.col('p.criado_em'), 'yyyyMMdd').cast('int').alias('data_id'),
        F.col('i.quantidade').alias('quantidade'),
        F.col('i.preco_unitario').alias('preco_unitario'),
        F.col('i.subtotal').alias('valor_bruto'),
        # rateio proporcional da taxa de entrega
        (F.col('p.taxa_entrega') *
         (F.col('i.subtotal') / F.col('p.subtotal'))).cast('decimal(10,2)').alias('taxa_entrega_alocada'),
        F.col('p.canal').alias('canal_venda'),
        F.col('pg.metodo').alias('metodo_pagamento'),
        F.col('p.criado_em').alias('pedido_timestamp'))
    .withColumn('valor_liquido', F.col('valor_bruto') + F.col('taxa_entrega_alocada')))

(fact.write
    .mode('overwrite')
    .partitionBy('data_id')
    .format('delta')
    .save(f'{BASE_GOLD}/fact_vendas'))

print(f'  fact_vendas: {fact.count():,} linhas')
print('\n=== GOLD OK ===')


Construindo fact_vendas...
  fact_vendas: 10,793 linhas

=== GOLD OK ===


In [0]:
# Faturamento por regiao
fact_g  = spark.read.format('delta').load(f'{BASE_GOLD}/fact_vendas')
dim_loj = spark.read.format('delta').load(f'{BASE_GOLD}/dim_loja')

print('Faturamento por regiao:')
(fact_g.join(dim_loj, 'loja_id')
    .groupBy('loja_regiao')
    .agg(F.sum('valor_bruto').alias('faturamento'),
         F.countDistinct('pedido_id').alias('pedidos'))
    .orderBy(F.desc('faturamento'))
    .show())

Faturamento por regiao:
+------------+-----------+-------+
| loja_regiao|faturamento|pedidos|
+------------+-----------+-------+
|     Sudeste|  162275.96|   2146|
|    Nordeste|   92101.06|   1148|
|         Sul|   56848.30|    739|
|Centro-Oeste|   27222.53|    335|
+------------+-----------+-------+

